<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/early_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
dataset_dir = "/content/drive/MyDrive/dataset"
labels_path = "/content/drive/MyDrive/results/labels_from_metadata.json"
embeddings_file = "/content/drive/MyDrive/fused_reduced.npz"
SPLIT_DIR = "/content/drive/MyDrive/splits (1)"

batch_size = 32
epochs = 20
hidden_dim = 128
learning_rate = 1e-3

SENTIMENTS = ["positive", "neutral", "negative"]
label_to_idx = {s: i for i, s in enumerate(SENTIMENTS)}
idx_to_label = {i: s for i, s in enumerate(SENTIMENTS)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
with open(labels_path, "r") as f:
    unimodal_data = json.load(f)

print("Loaded pseudo-labels:", len(unimodal_data))


In [ ]:
data = np.load(embeddings_file)
video_ids = data["video_ids"]
embeddings = data["embeddings"]

fused_embeddings_dict = {
    str(video_ids[i]): embeddings[i]
    for i in range(len(video_ids))
}

print("Loaded embeddings:", len(fused_embeddings_dict))

In [ ]:
train_ids = set(pd.read_csv(os.path.join(SPLIT_DIR, "train.csv"))["video_id"].astype(str))
val_ids   = set(pd.read_csv(os.path.join(SPLIT_DIR, "val.csv"))["video_id"].astype(str))
test_ids  = set(pd.read_csv(os.path.join(SPLIT_DIR, "test.csv"))["video_id"].astype(str))

print("Split sizes:", len(train_ids), len(val_ids), len(test_ids))

In [ ]:
X, y, video_list = [], [], []

for item in unimodal_data:
    vid = str(item["video_id"])

    if vid not in fused_embeddings_dict:
        continue

    label = item.get("label", "neutral").strip().lower()
    if label not in SENTIMENTS:
        label = "neutral"

    X.append(fused_embeddings_dict[vid])
    y.append(label_to_idx[label])
    video_list.append(vid)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
video_list = np.array(video_list)

print("Total usable samples:", len(y))

In [ ]:
def split_data(ids_set):
    indices = [i for i, vid in enumerate(video_list) if vid in ids_set]
    return X[indices], y[indices]

X_train, y_train = split_data(train_ids)
X_val, y_val     = split_data(val_ids)
X_test, y_test   = split_data(test_ids)

print("Train:", len(y_train), "Val:", len(y_val), "Test:", len(y_test))

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.tensor(embeddings)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

In [ ]:
train_loader = DataLoader(FusionDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(FusionDataset(X_val, y_val), batch_size=batch_size)
test_loader  = DataLoader(FusionDataset(X_test, y_test), batch_size=batch_size)


In [ ]:
class EarlyFusionSentiment(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [ ]:
embedding_dim = X_train.shape[1]

model = EarlyFusionSentiment(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Class weights:", class_weights)

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    total_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss:.4f}")

In [ ]:
def evaluate(loader):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())

    return np.array(all_preds), np.array(all_targets)

In [ ]:
test_preds, test_targets = evaluate(test_loader)

print("Accuracy:", accuracy_score(test_targets, test_preds))
print("Balanced Accuracy:", balanced_accuracy_score(test_targets, test_preds))
print("Macro F1:", f1_score(test_targets, test_preds, average="macro"))
print("Weighted F1:", f1_score(test_targets, test_preds, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(test_targets, test_preds, target_names=SENTIMENTS))

In [ ]:
cm = confusion_matrix(test_targets, test_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=SENTIMENTS,
            yticklabels=SENTIMENTS,
            cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
X_all = torch.tensor(X, dtype=torch.float32).to(device)
video_ids_all = video_list
y_all = y

def predict_all(model, X_tensor, batch_size=64):
    model.eval()
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            xb = X_tensor[i:i+batch_size]
            logits = model(xb)
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return np.array(all_preds), np.array(all_probs)

preds_all, probs_all = predict_all(model, X_all)

accuracy = accuracy_score(y_all, preds_all)
balanced_acc = balanced_accuracy_score(y_all, preds_all)
macro_f1 = f1_score(y_all, preds_all, average="macro")
weighted_f1 = f1_score(y_all, preds_all, average="weighted")

print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_all, preds_all, target_names=SENTIMENTS))

cm = confusion_matrix(y_all, preds_all)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=SENTIMENTS, yticklabels=SENTIMENTS, cmap="Blues")
plt.title("Confusion Matrix (Counts)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

cm_norm = confusion_matrix(y_all, preds_all, normalize="true")
plt.figure(figsize=(6,5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", xticklabels=SENTIMENTS, yticklabels=SENTIMENTS, cmap="Blues")
plt.title("Confusion Matrix (Normalized)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

final_results = []
for i, vid in enumerate(video_ids_all):
    final_results.append({
        "video_id": str(vid),
        "true_label": idx_to_label[y_all[i]],
        "predicted_label": idx_to_label[preds_all[i]],
        "confidence": float(np.max(probs_all[i])),
        "probabilities": probs_all[i].tolist()
    })

output_json = "/content/drive/MyDrive/early_fusion_full_predictions.json"
output_csv  = "/content/drive/MyDrive/early_fusion_full_predictions.csv"

# Save JSON
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)
print(f"Saved full dataset predictions to {output_json}")

# Save CSV
df_results = pd.DataFrame(final_results)
df_results.to_csv(output_csv, index=False)
print(f"Saved full dataset predictions to {output_csv}")